<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/05_TADP_Experiment_D_Governance_Evidence_Sensitivity_K10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [2]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

# ======================================================================================
# 05_TADP_Experiment_D_FINAL_A1A2BC_OptionA_Evidence_Sensitivity_K10.py
# TADP Experiment D — Documentary-evidence assignment sensitivity (FINAL A1/A2/B/C-aligned design)
# ======================================================================================
#
# SCIENTIFIC QUESTION
# -------------------
# Does TADP's governance outcome and downstream TADP-VR utility depend strongly on one
# particular controlled documentary-evidence assignment?
#
# FINAL DESIGN IMPLEMENTED HERE
# -----------------------------
# * K = 10, anchored to the main Experiment-A Diabetes split/partition protocol.
# * Data Quality (dim2) is measured ONCE from each fixed TRAIN client shard and is never
#   synthesized or changed across evidence realizations.
# * Documentary evidence uses a fixed library of 120 bundles:
#       6 designed policy-branch families x 20 variants each.
# * Each evidence realization independently samples 10 bundles WITH REPLACEMENT from the
#   120-bundle library. Duplicate bundles and unused bundles are therefore allowed.
# * Admission counts are NOT forced. They emerge from the sampled documentary evidence,
#   fixed machine-measured DQ, and the final TADP policy.
# * D1: 500 governance-only evidence assignments; no model training.
# * D2: 10 preregistered evidence assignments x 3 training seeds for TADP-VR utility.
# * Vanilla FedAvg is trained once per training seed as an evidence-independent contextual
#   reference. Random-K is intentionally NOT repeated here because the main Experiment A1
#   already provides the matched Random-K selection control.
# * If two D2 evidence assignments produce the same admitted cohort, that cohort is trained
#   only once per training seed and the exact result is reused. This is mathematically the
#   same downstream training experiment under the frozen data/model/W0 protocol.
#
# FINAL TADP POLICY
# -----------------
# evidence validity -> dimension floor -> HPS -> HPS_review -> binary ACCEPT/REJECT
# No Critical WAC, no Global Critical WAC, no critical-factor fallback, no TADP-SDA.
#
# LEAKAGE CONTROL
# ---------------
# GLOBAL patient-grouped holdout is created before client partitioning.
# DQ/governance operate on raw TRAIN shards. For each D2 admitted cohort,
# model preprocessing and class weights are fitted AFTER admission using only
# that cohort's TRAIN shards (Option A). TEST is transformed only after the
# scenario-specific preprocessing map is frozen and is never used for fitting.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
import re
from copy import deepcopy
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ======================================================================================
# FINAL TADP POLICY CONSTANTS
# ======================================================================================
GOOD_CUT = 3.0
HIGH_CUT = 3.5
DIMENSION_MIN_FLOOR = 2.5
REVIEW_MIDPOINT = (GOOD_CUT + HIGH_CUT) / 2.0  # 3.25
MAX_FACTOR_SCORE = 5.0

WEIGHTS_HPS = {
    "dim1": 0.25,
    "dim2": 0.15,
    "dim3": 0.10,
    "dim4": 0.10,
    "dim5": 0.30,
    "dim6": 0.10,
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

FACTOR_NAMES = {
    "dim1": ["source_reputation", "data_controller", "data_objective", "data_collection_lineage"],
    "dim2": [
        "completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
        "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": ["data_dictionary", "version_logs", "collection_protocol", "definition_updates"],
    "dim4": ["data_freshness", "scheduled_refresh", "retention_clarity"],
    "dim5": ["regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"],
    "dim6": ["license_terms", "ethical_reviews", "redistribution", "user_agreements"],
}
DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# Final A1/A2 governance policy — no factor-specific mandatory vetoes.
CRITICAL_REVIEW_DIMS = ("dim1", "dim2", "dim5", "dim6")
CRITICAL_REVIEW_WEIGHTS = {
    d: WEIGHTS_HPS[d] / sum(WEIGHTS_HPS[q] for q in CRITICAL_REVIEW_DIMS)
    for d in CRITICAL_REVIEW_DIMS
}
_EXPECTED_CRITICAL_REVIEW_WEIGHTS = {
    "dim1": 0.3125, "dim2": 0.1875, "dim5": 0.375, "dim6": 0.125,
}
assert all(math.isclose(CRITICAL_REVIEW_WEIGHTS[d], v, rel_tol=0.0, abs_tol=1e-12)
           for d, v in _EXPECTED_CRITICAL_REVIEW_WEIGHTS.items())
EVIDENCE_VALIDITY_MIN_EXCLUSIVE = 0.0

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

AVERAGE_COMPUTE_POWER_W = 12.0
ELECTRICITY_PRICE_USD_PER_KWH = 0.20
CARBON_INTENSITY_KG_PER_KWH = 0.445
COMMUNICATION_PRICE_USD_PER_MB = 0.005
MB = 1_000_000
MIB = 1024 ** 2

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]

# ======================================================================================
# EXPERIMENT D CONSTANTS — FINAL A1/A2/B/C OPTION-A DESIGN
# ======================================================================================
EXPERIMENT_VERSION = "TADP-D-FINAL-A1A2BC-28F-HPSREVIEW-OPTIONA-K10-500GOV-10EVID-3SEED-4ROUND"
K_SUBMISSIONS = 10
TRAINING_RUN_SEEDS = [42, 142, 242]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 100
GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
DOMAIN = "healthcare"
USE_GOOGLE_DRIVE_CHECKPOINTS = True

BRANCH_FAMILIES = [
    "DIRECT_ACCEPT",
    "HPS_REVIEW_ACCEPT",
    "HPS_REVIEW_REJECT",
    "DIMENSION_FLOOR_REJECT",
    "LOW_HPS_REJECT",
    "EVIDENCE_VALIDITY_REJECT",
]
LIBRARY_VARIANTS_PER_FAMILY = 20
DOCUMENTARY_LIBRARY_SIZE = len(BRANCH_FAMILIES) * LIBRARY_VARIANTS_PER_FAMILY
LIBRARY_REFERENCE_DQ_SCORE = 4.0

PREDICTIVE_EVIDENCE_SEEDS = [
    1042, 11042, 21042, 31042, 41042,
    51042, 61042, 71042, 81042, 91042,
]
_ADDITIONAL_GOVERNANCE_SEEDS = list(range(100042, 100532))
GOVERNANCE_SWEEP_SEEDS = PREDICTIVE_EVIDENCE_SEEDS + _ADDITIONAL_GOVERNANCE_SEEDS
assert len(GOVERNANCE_SWEEP_SEEDS) == 500
assert len(set(GOVERNANCE_SWEEP_SEEDS)) == 500
assert abs(sum(WEIGHTS_HPS.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert DOCUMENTARY_LIBRARY_SIZE == 120

RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])

def dimension_scores_from_factors(factors: Dict[str, Dict[str, float]]) -> Dict[str, float]:
    out = {}
    for dim in FACTOR_NAMES:
        vals = [float(factors[dim][f]) for f in FACTOR_NAMES[dim]]
        out[dim] = float(np.mean(vals))
    return out

def compute_hps(dimensions: Dict[str, float]) -> float:
    return float(sum(WEIGHTS_HPS[d] * float(dimensions[d]) for d in WEIGHTS_HPS))


def evidence_validity_audit(
    factors: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    """
    Final A1/A2/B/C evidence-validity gate.

    Every one of the 28 factor scores must be present, numeric, finite, and
    strictly greater than zero. A failure is non-compensable and rejects the
    contribution before dimension-floor or HPS evaluation.
    """
    missing, nonfinite, nonpositive = [], [], []
    scores = {}

    for dim, expected_factors in FACTOR_NAMES.items():
        for factor in expected_factors:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue
            try:
                value = float(factors[dim][factor])
            except Exception:
                nonfinite.append(key)
                continue
            if not np.isfinite(value):
                nonfinite.append(key)
                continue
            scores[key] = value
            if value <= EVIDENCE_VALIDITY_MIN_EXCLUSIVE:
                nonpositive.append(f"{key}:{value:.3f}")

    expected_n = int(sum(len(v) for v in FACTOR_NAMES.values()))
    passed = (
        len(scores) == expected_n
        and not missing
        and not nonfinite
        and not nonpositive
    )
    return {
        "passed": bool(passed),
        "factor_count_expected": expected_n,
        "factor_count_observed_valid_numeric": int(len(scores)),
        "missing": missing,
        "nonfinite": nonfinite,
        "nonpositive": nonpositive,
    }


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    return [
        dim
        for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def compute_hps_review(
    dimensions: Dict[str, float],
) -> float:
    """
    Critical-dimension automated review used only in the HPS borderline band.

    HPS_review uses D1, D2, D5, and D6 with the ORIGINAL HPS weights
    renormalized over those four dimensions.
    """
    vals = [float(dimensions[d]) for d in CRITICAL_REVIEW_DIMS]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(
        sum(
            CRITICAL_REVIEW_WEIGHTS[d] * float(dimensions[d])
            for d in CRITICAL_REVIEW_DIMS
        )
    )


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str = DOMAIN,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    FINAL frozen A1/A2/B/C TADP admission policy.

    1) All 28 factors present, finite, and >0; otherwise Reject.
    2) Every dimension >=2.5; otherwise Reject.
    3) HPS over all six dimensions:
         HPS <3.0  -> Reject
         HPS >=3.5 -> Direct Accept
         [3.0,3.5) -> Automated Review
    4) Automated review uses HPS_review over D1,D2,D5,D6:
         HPS_review >=3.25 -> Accept
         otherwise         -> Reject
    """
    domain = str(domain).lower()

    validity = evidence_validity_audit(factors)
    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)
    hps_review = compute_hps_review(dims)
    dim_floor_failed = dimension_floor_failures(
        dims, floor=DIMENSION_MIN_FLOOR
    )

    base = {
        "hps": float(hps) if np.isfinite(hps) else np.nan,
        "hps_review": float(hps_review) if np.isfinite(hps_review) else np.nan,
        "hps_review_midpoint": float(REVIEW_MIDPOINT),
        "hps_review_margin": (
            float(hps_review - REVIEW_MIDPOINT)
            if np.isfinite(hps_review) else np.nan
        ),
        "critical_review_dimensions": ";".join(CRITICAL_REVIEW_DIMS),
        "critical_review_weights_json": json.dumps(
            CRITICAL_REVIEW_WEIGHTS, sort_keys=True
        ),
        "dimensions": dims,
        "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures": ";".join(dim_floor_failed),
        "evidence_validity_pass": bool(validity["passed"]),
        "evidence_factor_count_expected": int(
            validity["factor_count_expected"]
        ),
        "evidence_factor_count_observed_valid_numeric": int(
            validity["factor_count_observed_valid_numeric"]
        ),
        "evidence_missing": ";".join(validity["missing"]),
        "evidence_nonfinite": ";".join(validity["nonfinite"]),
        "evidence_nonpositive": ";".join(validity["nonpositive"]),
    }

    if not validity["passed"]:
        failures = (
            validity["missing"]
            + validity["nonfinite"]
            + validity["nonpositive"]
        )
        return {
            **base,
            "decision_path": "EVIDENCE_VALIDITY_GATE",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_EVIDENCE_VALIDITY",
            "reason": (
                "At least one of the 28 factor scores is missing, non-finite, "
                "or <=0: " + ";".join(failures)
            ),
        }

    if dim_floor_failed:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    if hps < float(good_cut):
        return {
            **base,
            "decision_path": "LOW_HPS",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
            "reason": f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    if hps >= float(high_cut):
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "initial_action": "AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
            "reason": (
                "All 28 factor scores valid; every dimension >= "
                f"{DIMENSION_MIN_FLOOR:.1f}; HPS {hps:.3f} >= "
                f"{high_cut:.3f}"
            ),
        }

    review_pass = bool(
        np.isfinite(hps_review)
        and float(hps_review) >= float(REVIEW_MIDPOINT)
    )

    if review_pass:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "initial_action": "AUTOMATED_REVIEW",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
                f"HPS_review {hps_review:.3f} >= midpoint "
                f"{REVIEW_MIDPOINT:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "initial_action": "AUTOMATED_REVIEW",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_HPS_REVIEW",
        "reason": (
            f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
            f"HPS_review {hps_review:.3f} < midpoint "
            f"{REVIEW_MIDPOINT:.3f}"
        ),
    }



def _documentary_profile(
    role: str,
    variant: int = 0,
) -> Dict[str, Dict[str, float]]:
    """
    Controlled documentary profiles for Experiment D.

    Only the 20 documentary factors are synthesized. DQ (D2) is always
    machine-measured from the fixed raw TRAIN client shard.

    Profiles are designed to exercise one of the six final-policy branches
    when paired with the structural reference DQ score. Actual client DQ can
    legitimately alter the observed branch in a realization.
    """
    x = {
        dim: {f: 4.0 for f in FACTOR_NAMES[dim]}
        for dim in DOCUMENTARY_DIMS
    }

    if role == "DIRECT_ACCEPT":
        # Strong valid evidence; comfortably above direct-accept HPS.
        pass

    elif role == "HPS_REVIEW_ACCEPT":
        # D1=3.5, D3=3.0, D4=3.0, D5=3.2, D6=3.0.
        # With reference D2=4.0: HPS is in [3.0,3.5), HPS_review >=3.25.
        x["dim1"] = {
            "source_reputation": 3.0,
            "data_controller": 4.0,
            "data_objective": 3.0,
            "data_collection_lineage": 4.0,
        }
        x["dim3"] = {f: 3.0 for f in FACTOR_NAMES["dim3"]}
        x["dim4"] = {f: 3.0 for f in FACTOR_NAMES["dim4"]}
        x["dim5"] = {
            "regulation_coverage": 3.0,
            "consent_ethics": 3.0,
            "geo_restrictions": 3.0,
            "sensitivity_classification": 3.0,
            "audits": 4.0,
        }
        x["dim6"] = {f: 3.0 for f in FACTOR_NAMES["dim6"]}

    elif role == "HPS_REVIEW_REJECT":
        # Critical dimensions are intentionally weaker while noncritical
        # D3/D4 keep the global HPS in the automated-review band.
        x["dim1"] = {
            "source_reputation": 2.0,
            "data_controller": 3.0,
            "data_objective": 2.0,
            "data_collection_lineage": 3.0,
        }  # mean 2.5
        x["dim3"] = {f: 5.0 for f in FACTOR_NAMES["dim3"]}
        x["dim4"] = {f: 5.0 for f in FACTOR_NAMES["dim4"]}
        x["dim5"] = {
            "regulation_coverage": 2.0,
            "consent_ethics": 3.0,
            "geo_restrictions": 2.0,
            "sensitivity_classification": 3.0,
            "audits": 3.0,
        }  # mean 2.6
        x["dim6"] = {
            "license_terms": 2.0,
            "ethical_reviews": 3.0,
            "redistribution": 2.0,
            "user_agreements": 3.0,
        }  # mean 2.5

    elif role == "DIMENSION_FLOOR_REJECT":
        # Valid factors, but D4 mean is below 2.5.
        x["dim4"] = {f: 2.0 for f in FACTOR_NAMES["dim4"]}

    elif role == "LOW_HPS_REJECT":
        # All dimensions remain >=2.5, while global HPS stays below 3.0.
        x["dim1"] = {
            "source_reputation": 2.0,
            "data_controller": 3.0,
            "data_objective": 2.0,
            "data_collection_lineage": 3.0,
        }
        x["dim3"] = {
            "data_dictionary": 2.0,
            "version_logs": 3.0,
            "collection_protocol": 2.0,
            "definition_updates": 3.0,
        }
        x["dim4"] = {
            "data_freshness": 2.0,
            "scheduled_refresh": 3.0,
            "retention_clarity": 3.0,
        }
        x["dim5"] = {
            "regulation_coverage": 2.0,
            "consent_ethics": 3.0,
            "geo_restrictions": 2.0,
            "sensitivity_classification": 3.0,
            "audits": 3.0,
        }
        x["dim6"] = {
            "license_terms": 2.0,
            "ethical_reviews": 3.0,
            "redistribution": 2.0,
            "user_agreements": 3.0,
        }

    elif role == "EVIDENCE_VALIDITY_REJECT":
        # Non-compensable final-policy failure: factor score must be >0.
        x["dim1"]["source_reputation"] = 0.0

    else:
        raise ValueError(f"Unknown controlled evidence role: {role}")

    return x



def build_tadp_governance(
    client_ids,
    documentary_evidence,
    dq_scores,
    evidence_seed: int,
) -> pd.DataFrame:
    rows = []

    for cid in client_ids:
        factors = {
            d: dict(documentary_evidence[cid][d])
            for d in DOCUMENTARY_DIMS
        }
        factors["dim2"] = {
            f: float(dq_scores[cid][f])
            for f in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(factors, domain=DOMAIN)

        row = {
            "client": str(cid),
            "evidence_seed": int(evidence_seed),
            "hps": d["hps"],
            "hps_review": d["hps_review"],
            "hps_review_midpoint": d["hps_review_midpoint"],
            "hps_review_margin": d["hps_review_margin"],
            "critical_review_dimensions": d[
                "critical_review_dimensions"
            ],
            "critical_review_weights_json": d[
                "critical_review_weights_json"
            ],
            "evidence_validity_pass": bool(
                d["evidence_validity_pass"]
            ),
            "evidence_factor_count_expected": int(
                d["evidence_factor_count_expected"]
            ),
            "evidence_factor_count_observed_valid_numeric": int(
                d["evidence_factor_count_observed_valid_numeric"]
            ),
            "evidence_missing": d["evidence_missing"],
            "evidence_nonfinite": d["evidence_nonfinite"],
            "evidence_nonpositive": d["evidence_nonpositive"],
            "dimension_min_floor": d["dimension_min_floor"],
            "dimension_floor_failures": d[
                "dimension_floor_failures"
            ],
            "decision_path": d["decision_path"],
            "initial_action": d["initial_action"],
            "final_action": d["final_action"],
            "status": d["status"],
            "reason": d["reason"],
        }

        for dim, value in d["dimensions"].items():
            row[f"{dim}_score_0_5"] = (
                float(value) if np.isfinite(value) else np.nan
            )

        rows.append(row)

    return pd.DataFrame(rows)


def accepted_tadp_vr(
    governance_df: pd.DataFrame,
) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def policy_self_test():
    expected = {
        "DIRECT_ACCEPT": ("DIRECT_AUTO_ACCEPT", "ACCEPT"),
        "HPS_REVIEW_ACCEPT": ("HPS_REVIEW_BAND", "ACCEPT"),
        "HPS_REVIEW_REJECT": ("HPS_REVIEW_BAND", "REJECT"),
        "DIMENSION_FLOOR_REJECT": ("DIMENSION_FLOOR", "REJECT"),
        "LOW_HPS_REJECT": ("LOW_HPS", "REJECT"),
        "EVIDENCE_VALIDITY_REJECT": (
            "EVIDENCE_VALIDITY_GATE", "REJECT"
        ),
    }

    for role, (path, action) in expected.items():
        factors = {
            d: dict(_documentary_profile(role)[d])
            for d in DOCUMENTARY_DIMS
        }
        factors["dim2"] = {
            f: LIBRARY_REFERENCE_DQ_SCORE
            for f in FACTOR_NAMES["dim2"]
        }
        got = tadp_decision(factors, domain=DOMAIN)

        if (
            got["decision_path"],
            got["final_action"],
        ) != (path, action):
            raise RuntimeError(
                f"Policy self-test failed for {role}: {got}"
            )

    if not math.isclose(
        sum(CRITICAL_REVIEW_WEIGHTS.values()),
        1.0,
        rel_tol=0.0,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            "Critical review weights do not sum to 1."
        )

    return True


def print_banner(title: str, width: int = 108):
    """Compact console section banner used by the Colab experiment driver."""
    print("\n" + "=" * int(width))
    print(str(title))
    print("=" * int(width))

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()

def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()

def _process_rss_mib() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0

class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mib = 0.0
        self.end_mib = 0.0
        self.peak_mib = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mib = max(self.peak_mib, _process_rss_mib())

    def start(self):
        self.start_mib = _process_rss_mib()
        self.peak_mib = self.start_mib
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mib = _process_rss_mib()
        self.peak_mib = max(self.peak_mib, self.start_mib, self.end_mib)
        return {
            "ram_start_mib": float(self.start_mib),
            "ram_end_mib": float(self.end_mib),
            "ram_peak_mib": float(self.peak_mib),
            "ram_delta_mib": float(max(0.0, self.peak_mib - self.start_mib)),
            "ram_mib": float(self.peak_mib),
        }

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)

def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )

def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MiB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path

def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0

def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0

def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}

def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))

def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out

def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = 4096,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]

def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"

def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mib: float,
    ram_start_mib: float,
    ram_peak_mib: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        AVERAGE_COMPUTE_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mib) - float(ram_start_mib))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mib:.3f} MiB | "
        f"RAM peak={ram_peak_mib:.1f} MiB | RAM Δ={ram_delta:.1f} MiB"
    )
    print("-" * 112)

def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))

def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))

def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out

def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mib = float(cumulative_comm_raw_b / MIB)

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mib)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                AVERAGE_COMPUTE_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mib": float(cumulative_comm_mib),
            "ram_peak_mib_live": float(live_peak),
            "ram_delta_mib_live": float(max(0.0, live_peak - monitor.start_mib)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mib=cumulative_comm_mib,
            ram_start_mib=float(monitor.start_mib),
            ram_peak_mib=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_bytes": int(communication_b),
        "communication_mb": float(communication_b / MB),
        "communication_mib": float(communication_b / MIB),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }

def result_row(run: int, seed: int, scenario: str, result: Dict[str, Any], initial_hash: str) -> Dict[str, Any]:
    row = {
        "run": int(run), "seed": int(seed), "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_bytes": int(result.get("communication_bytes", 0)),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mib": float(result.get("ram_start_mib", 0.0)),
        "ram_end_mib": float(result.get("ram_end_mib", 0.0)),
        "ram_peak_mib": float(result.get("ram_peak_mib", 0.0)),
        "ram_delta_mib": float(result.get("ram_delta_mib", 0.0)),
        "initial_weights_sha256": str(initial_hash),
    }
    row["estimated_energy_wh"] = AVERAGE_COMPUTE_POWER_W * row["runtime_s"] / 3600.0
    row["estimated_energy_kwh"] = row["estimated_energy_wh"] / 1000.0
    row["estimated_co2_kg"] = row["estimated_energy_kwh"] * CARBON_INTENSITY_KG_PER_KWH
    row["estimated_energy_cost_usd"] = row["estimated_energy_kwh"] * ELECTRICITY_PRICE_USD_PER_KWH
    row["communication_cost_usd"] = float(result.get("communication_mb", row["communication_mib"] * MIB / MB)) * COMMUNICATION_PRICE_USD_PER_MB
    row["estimated_total_cost_usd"] = row["estimated_energy_cost_usd"] + row["communication_cost_usd"]
    return row

def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )

def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df

def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap

@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)

if not hasattr(TabularPreprocessor, "__dataclass_fields__"):
    raise RuntimeError("TabularPreprocessor must be decorated with @dataclass.")
if not callable(getattr(TabularPreprocessor, "transform", None)):
    raise RuntimeError("TabularPreprocessor.transform() is missing from the class definition.")

def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )

def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root

def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)

def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))

def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)

def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)

def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


# ======================================================================================
# EXPERIMENT-D DATA PREPARATION — EXACT K=10 MAIN-EXPERIMENT ANCHOR
# ======================================================================================

def dirichlet_partition_dataframe_k10(
    train_df: pd.DataFrame,
    alpha: float,
    seed: int,
    min_client_records: int = MIN_CLIENT_RECORDS,
) -> Dict[str, pd.DataFrame]:
    """Exact K=10 label-wise Dirichlet logic with stable client IDs A--J."""
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJ")

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(K_SUBMISSIONS)]
        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(K_SUBMISSIONS, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }
    raise RuntimeError("Could not obtain the fixed K=10 Dirichlet partition.")


def row_level_leakage_audit(train_df, test_df, clients) -> Dict[str, Any]:
    train_ids = set(train_df["_row_id"].astype(int).tolist())
    test_ids = set(test_df["_row_id"].astype(int).tolist())
    all_client_ids = []
    for df in clients.values():
        all_client_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_client_ids)
    audit = {
        "train_test_row_overlap": int(len(train_ids & test_ids)),
        "test_rows_in_any_client": int(len(test_ids & client_set)),
        "train_rows_missing_from_clients": int(len(train_ids - client_set)),
        "client_rows_not_in_global_train": int(len(client_set - train_ids)),
        "duplicate_train_rows_across_clients": int(len(all_client_ids) - len(client_set)),
    }
    audit["pass"] = bool(all(v == 0 for v in audit.values()))
    return audit


@dataclass
class TabularDQSchema:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]

def infer_train_only_dq_schema(client_frames: Dict[str,pd.DataFrame]) -> TabularDQSchema:
    any_df=next(iter(client_frames.values()))
    feature_cols=[c for c in any_df.columns if c not in {TARGET,"_target","_row_id",*ID_COLUMNS}]
    numeric_cols=[]; categorical_cols=[]
    for c in feature_cols:
        total=0; valid=0
        for df in client_frames.values():
            raw=df[c]; nm=raw.notna(); total += int(nm.sum())
            if nm.any(): valid += int(pd.to_numeric(raw[nm],errors="coerce").notna().sum())
        (numeric_cols if valid/max(1,total)>=0.95 else categorical_cols).append(c)
    return TabularDQSchema(feature_cols,numeric_cols,categorical_cols)

def preprocessor_fingerprint(pre: TabularPreprocessor) -> str:
    payload={"feature_cols":pre.feature_cols,"numeric_cols":pre.numeric_cols,"categorical_cols":pre.categorical_cols,
             "mean":pre.mean,"std":pre.std,"categories":pre.categories}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()

def build_post_selection_training_view(clients_raw,test_df,participating_clients,view_name):
    ids=sorted(map(str,participating_clients))
    if not ids: raise ValueError("Cannot fit preprocessing on an empty participating cohort")
    selected={cid:clients_raw[cid] for cid in ids}
    pre=fit_federated_train_only_preprocessor(selected)
    arrays={cid:(pre.transform(selected[cid]),selected[cid]["_target"].to_numpy(dtype=np.int32)) for cid in ids}
    X_test=pre.transform(test_df); y_test=test_df["_target"].to_numpy(dtype=np.int32)
    y_all=np.concatenate([arrays[c][1] for c in ids])
    return {"view_name":str(view_name),"participating_clients":ids,"preprocessor":pre,"preprocessor_sha256":preprocessor_fingerprint(pre),
            "client_arrays":arrays,"X_test":X_test,"y_test":y_test,"class_weights":class_weight_dict(y_all),
            "input_dim":int(X_test.shape[1])}

def prepare_main_k10(csv_path: str) -> Dict[str, Any]:
    raw=load_diabetes(csv_path)
    train_df,test_df,patient_overlap=global_patient_grouped_split(raw,GLOBAL_SPLIT_SEED)
    clients=dirichlet_partition_dataframe_k10(train_df,alpha=DIRICHLET_ALPHA,seed=CLIENT_PARTITION_SEED)
    if list(clients.keys()) != list("ABCDEFGHIJ"): raise RuntimeError("Main K=10 client IDs are not A--J")
    leakage=row_level_leakage_audit(train_df,test_df,clients); leakage["patient_overlap"]=int(patient_overlap)
    leakage["pass"]=bool(leakage["pass"] and patient_overlap==0)
    if not leakage["pass"]: raise RuntimeError(f"Leakage audit failed: {leakage}")
    dq_schema=infer_train_only_dq_schema(clients)
    reference=build_tabular_reference(clients,dq_schema)
    dq_scores={}; dq_rows=[]
    for cid,df in clients.items():
        scores,raw_metrics=tabular_dq_scores(df,dq_schema,reference); dq_scores[cid]=scores; dq_rows.append({"client":cid,**scores,**raw_metrics})
    part=[]
    for cid,df in clients.items():
        counts=df["_target"].value_counts().to_dict(); part.append({"client":cid,"records":len(df),"class_0_NO":counts.get(0,0),"class_1_GT30":counts.get(1,0),"class_2_LT30":counts.get(2,0)})
    return {"raw":raw,"train_df":train_df,"test_df":test_df,"clients_raw":clients,"client_ids":list(clients),
            "dq_schema":dq_schema,"dq_reference":reference,"dq_scores":dq_scores,"dq_audit":pd.DataFrame(dq_rows),
            "partition_table":pd.DataFrame(part),"leakage_audit":leakage,"train_rows":len(train_df),"test_rows":len(test_df)}



# ======================================================================================
# FIXED 120-BUNDLE DOCUMENTARY-EVIDENCE LIBRARY
# ======================================================================================

def _profile_signature(profile: Dict[str, Dict[str, float]]) -> Tuple[float, ...]:
    return tuple(
        float(profile[dim][factor])
        for dim in DOCUMENTARY_DIMS
        for factor in FACTOR_NAMES[dim]
    )


def _with_reference_dq(profile: Dict[str, Dict[str, float]]) -> Dict[str, Dict[str, float]]:
    factors = {d: dict(profile[d]) for d in DOCUMENTARY_DIMS}
    factors["dim2"] = {
        f: float(LIBRARY_REFERENCE_DQ_SCORE) for f in FACTOR_NAMES["dim2"]
    }
    return factors


def _floor_seed_profile(target_dim: str) -> Dict[str, Dict[str, float]]:
    x=_documentary_profile("DIMENSION_FLOOR_REJECT")
    if target_dim in DOCUMENTARY_DIMS: x[target_dim]={f:2.0 for f in FACTOR_NAMES[target_dim]}
    return x




def _review_swap_candidates(profile: Dict[str, Dict[str, float]]):
    """Zero-sum within-dimension integer perturbations preserve dimension means and policy branch."""
    for dim in DOCUMENTARY_DIMS:
        names=list(FACTOR_NAMES[dim])
        for i in range(len(names)):
            for j in range(i+1,len(names)):
                a,b=names[i],names[j]
                for delta in (1.0,-1.0):
                    va=float(profile[dim][a]+delta); vb=float(profile[dim][b]-delta)
                    if 0.0 <= va <= 5.0 and 0.0 <= vb <= 5.0:
                        cand=deepcopy(profile); cand[dim][a]=va; cand[dim][b]=vb; yield cand



def _family_seed_profiles(
    family: str,
) -> List[Dict[str, Dict[str, float]]]:
    return [_documentary_profile(family, 0)]


def _branch_matches_family(
    decision: Dict[str, Any],
    family: str,
) -> bool:
    expected = {
        "DIRECT_ACCEPT": ("DIRECT_AUTO_ACCEPT", "ACCEPT"),
        "HPS_REVIEW_ACCEPT": ("HPS_REVIEW_BAND", "ACCEPT"),
        "HPS_REVIEW_REJECT": ("HPS_REVIEW_BAND", "REJECT"),
        "DIMENSION_FLOOR_REJECT": ("DIMENSION_FLOOR", "REJECT"),
        "LOW_HPS_REJECT": ("LOW_HPS", "REJECT"),
        "EVIDENCE_VALIDITY_REJECT": (
            "EVIDENCE_VALIDITY_GATE", "REJECT"
        ),
    }
    return (
        decision["decision_path"],
        decision["final_action"],
    ) == expected[family]


def _generate_family_variants(family: str, target_n: int) -> List[Dict[str, Dict[str, float]]]:
    queue=[deepcopy(x) for x in _family_seed_profiles(family)]; seen=set(); accepted=[]; qpos=0
    while qpos < len(queue) and len(accepted) < int(target_n):
        current=queue[qpos]; qpos += 1
        sig=_profile_signature(current)
        if sig in seen: continue
        seen.add(sig)
        decision=tadp_decision(_with_reference_dq(current))
        if _branch_matches_family(decision,family): accepted.append(current)
        for cand in _review_swap_candidates(current):
            sig2=_profile_signature(cand)
            if sig2 not in seen: queue.append(cand)
        if len(queue)>10000: break
    if len(accepted)<int(target_n): raise RuntimeError(f"Could generate only {len(accepted)} variants for {family}")
    return accepted[:int(target_n)]



def build_documentary_bundle_library():
    library={}; factor_rows=[]; bundle_rows=[]
    for family in BRANCH_FAMILIES:
        variants=_generate_family_variants(family,LIBRARY_VARIANTS_PER_FAMILY)
        for v,profile in enumerate(variants,start=1):
            bundle_id=f"{family}-V{v:02d}"
            library[bundle_id]={"bundle_id":bundle_id,"designed_branch_family":family,"library_variant":v,
                                "factors":{d:dict(profile[d]) for d in DOCUMENTARY_DIMS}}
            dec=tadp_decision(_with_reference_dq(profile))
            bundle_rows.append({"bundle_id":bundle_id,"designed_branch_family":family,"library_variant":v,
                                "reference_dq_score":LIBRARY_REFERENCE_DQ_SCORE,"reference_hps":dec["hps"],
                                "reference_hps_review":dec["hps_review"],"reference_evidence_validity_pass":dec["evidence_validity_pass"],
                                "reference_dimension_floor_failures":dec["dimension_floor_failures"],
                                "reference_decision_path":dec["decision_path"],"reference_final_action":dec["final_action"]})
            for dim in DOCUMENTARY_DIMS:
                for factor,score in profile[dim].items():
                    factor_rows.append({"bundle_id":bundle_id,"designed_branch_family":family,"library_variant":v,
                                        "dimension":dim,"dimension_name":DIMENSION_NAMES[dim],"factor":factor,
                                        "verified_rubric_level_0_5":float(score),"server_mapped_score_0_5":float(score),
                                        "rubric_descriptor":rubric_descriptor(dim,factor,score),
                                        "is_critical_review_dimension":dim in CRITICAL_REVIEW_DIMS,
                                        "critical_review_weight":CRITICAL_REVIEW_WEIGHTS.get(dim,np.nan),
                                        "evidence_validity_requirement":"present_finite_and_gt_zero",
                                        "human_role":"VERIFY_EVIDENCE_ONLY","admission_decision_by":"SERVER_POLICY"})
    bundles_df=pd.DataFrame(bundle_rows); factors_df=pd.DataFrame(factor_rows)
    counts=bundles_df["designed_branch_family"].value_counts().to_dict()
    expected={f:LIBRARY_VARIANTS_PER_FAMILY for f in BRANCH_FAMILIES}
    if len(library)!=DOCUMENTARY_LIBRARY_SIZE or counts!=expected: raise RuntimeError("Library balance failed")
    if not all(_branch_matches_family(tadp_decision(_with_reference_dq(library[r.bundle_id]["factors"])),r.designed_branch_family)
               for r in bundles_df.itertuples()): raise RuntimeError("Library branch validation failed")
    return library,bundles_df,factors_df



# ======================================================================================
# EVIDENCE REALIZATION: INDEPENDENT WITH-REPLACEMENT SAMPLING
# ======================================================================================

def sample_documentary_evidence_realization(
    client_ids: List[str],
    library: Dict[str, Dict[str, Any]],
    evidence_seed: int,
):
    rng = np.random.default_rng(int(evidence_seed))
    bundle_ids = sorted(library.keys())
    sampled = rng.choice(bundle_ids, size=len(client_ids), replace=True).tolist()

    documentary = {}
    assignment_rows = []
    for cid, bundle_id in zip(client_ids, sampled):
        bundle = library[str(bundle_id)]
        documentary[str(cid)] = {
            d: dict(bundle["factors"][d]) for d in DOCUMENTARY_DIMS
        }
        assignment_rows.append({
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "bundle_id": str(bundle_id),
            "designed_branch_family": bundle["designed_branch_family"],
            "library_variant": int(bundle["library_variant"]),
        })

    assignment = pd.DataFrame(assignment_rows)
    return documentary, assignment


def build_evidence_realization(
    client_ids: List[str],
    dq_scores: Dict[str, Dict[str, float]],
    library: Dict[str, Dict[str, Any]],
    evidence_seed: int,
):
    documentary, assignment = sample_documentary_evidence_realization(
        client_ids, library, evidence_seed
    )
    gov = build_tadp_governance(
        client_ids, documentary, dq_scores, evidence_seed=int(evidence_seed)
    ).sort_values("client").reset_index(drop=True)
    gov = gov.merge(assignment, on=["evidence_seed", "client"], how="left", validate="one_to_one")
    admitted = sorted(accepted_tadp_vr(gov))
    cohort_key = ";".join(admitted)
    cohort_hash = hashlib.sha256(cohort_key.encode("utf-8")).hexdigest()[:16]
    return documentary, assignment, gov, admitted, cohort_hash


# ======================================================================================
# D1 — 500 GOVERNANCE-ONLY ASSIGNMENTS
# ======================================================================================

def run_d1_governance_sensitivity(client_ids, dq_scores, library, out_dir: Path):
    realization_rows = []
    client_rows = []
    admitted_sets = {}

    for idx, evidence_seed in enumerate(GOVERNANCE_SWEEP_SEEDS, start=1):
        _, assignment, gov, admitted, cohort_hash = build_evidence_realization(
            client_ids, dq_scores, library, evidence_seed
        )
        aset = set(admitted)
        admitted_sets[int(evidence_seed)] = aset

        row = {
            "realization": int(idx),
            "evidence_seed": int(evidence_seed),
            "n_submissions": int(len(client_ids)),
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(len(admitted) / len(client_ids)),
            "admitted_clients": ";".join(admitted),
            "cohort_hash": cohort_hash,
            "unique_bundles_sampled": int(assignment["bundle_id"].nunique()),
            "duplicate_bundle_assignments": int(len(assignment) - assignment["bundle_id"].nunique()),
            "hps_mean": float(gov["hps"].mean()),
            "hps_sd_across_clients": float(gov["hps"].std(ddof=1)),
            "hps_review_mean": float(gov["hps_review"].mean()),
        }
        for family, count in assignment["designed_branch_family"].value_counts().items():
            row[f"sampled_family_count_{family.lower()}"] = int(count)
        for path, count in gov["decision_path"].value_counts().items():
            row[f"observed_path_count_{str(path).lower()}"] = int(count)
        realization_rows.append(row)

        merged = gov.copy()
        merged.insert(0, "realization", int(idx))
        for _, g in merged.iterrows():
            client_rows.append({
                "realization": int(idx),
                "evidence_seed": int(evidence_seed),
                "client": str(g["client"]),
                "bundle_id": str(g["bundle_id"]),
                "designed_branch_family": str(g["designed_branch_family"]),
                "observed_decision_path": str(g["decision_path"]),
                "final_action": str(g["final_action"]),
                "admitted": int(str(g["final_action"]) == "ACCEPT"),
                "hps": float(g["hps"]),
                "hps_review": float(g["hps_review"]),
                "evidence_validity_pass": bool(g["evidence_validity_pass"]),
                "dimension_floor_failures": str(g["dimension_floor_failures"]),
            })

        if idx % 50 == 0:
            print(f"D1 governance-only assignments: {idx}/{len(GOVERNANCE_SWEEP_SEEDS)} complete")

    realizations = pd.DataFrame(realization_rows)
    per_client = pd.DataFrame(client_rows)

    client_summary_rows = []
    for cid in client_ids:
        d = per_client[per_client["client"].eq(cid)]
        client_summary_rows.append({
            "client": str(cid),
            "n_realizations": int(len(d)),
            "admission_frequency": int(d["admitted"].sum()),
            "admission_rate": float(d["admitted"].mean()),
            "hps_mean": float(d["hps"].mean()),
            "hps_sd": float(d["hps"].std(ddof=1)),
            "hps_review_mean": float(d["hps_review"].mean()),
            "unique_bundle_count": int(d["bundle_id"].nunique()),
        })
    client_summary = pd.DataFrame(client_summary_rows)

    # Pairwise cohort stability across all 500 realizations.
    pair_rows = []
    seeds = list(admitted_sets.keys())
    for i in range(len(seeds)):
        a = admitted_sets[seeds[i]]
        for j in range(i + 1, len(seeds)):
            b = admitted_sets[seeds[j]]
            union = a | b
            inter = a & b
            pair_rows.append({
                "evidence_seed_a": int(seeds[i]),
                "evidence_seed_b": int(seeds[j]),
                "intersection_size": int(len(inter)),
                "union_size": int(len(union)),
                "jaccard_similarity": float(len(inter) / len(union)) if union else 1.0,
            })
    pairwise = pd.DataFrame(pair_rows)

    cohort_freq = (
        realizations.groupby(["cohort_hash", "admitted_clients", "accepted_count"], as_index=False)
        .size().rename(columns={"size": "realization_frequency"})
        .sort_values(["realization_frequency", "accepted_count"], ascending=[False, False])
        .reset_index(drop=True)
    )

    observed_paths = (
        per_client.groupby(["observed_decision_path", "final_action"], as_index=False)
        .size().rename(columns={"size": "count"})
    )

    j = pairwise["jaccard_similarity"].to_numpy(dtype=float)
    summary = {
        "n_governance_realizations": int(len(realizations)),
        "n_client_governance_decisions": int(len(per_client)),
        "accepted_count_min": int(realizations["accepted_count"].min()),
        "accepted_count_max": int(realizations["accepted_count"].max()),
        "accepted_count_mean": float(realizations["accepted_count"].mean()),
        "accepted_count_sd": float(realizations["accepted_count"].std(ddof=1)),
        "accepted_rate_mean": float(realizations["accepted_rate"].mean()),
        "unique_admitted_cohorts": int(realizations["cohort_hash"].nunique()),
        "empty_admitted_cohort_realizations": int((realizations["accepted_count"] == 0).sum()),
        "all_client_admitted_realizations": int((realizations["accepted_count"] == len(client_ids)).sum()),
        "client_admission_rate_min": float(client_summary["admission_rate"].min()),
        "client_admission_rate_max": float(client_summary["admission_rate"].max()),
        "pairwise_jaccard_pairs": int(len(pairwise)),
        "pairwise_jaccard_mean": float(np.mean(j)),
        "pairwise_jaccard_sd": float(np.std(j, ddof=1)),
        "pairwise_jaccard_min": float(np.min(j)),
        "pairwise_jaccard_max": float(np.max(j)),
    }

    realizations.to_csv(out_dir / "D1_governance_realizations_500.csv", index=False)
    per_client.to_csv(out_dir / "D1_client_decisions_500x10.csv", index=False)
    client_summary.to_csv(out_dir / "D1_client_admission_stability.csv", index=False)
    pairwise.to_csv(out_dir / "D1_pairwise_admitted_set_jaccard.csv", index=False)
    cohort_freq.to_csv(out_dir / "D1_admitted_cohort_frequency.csv", index=False)
    observed_paths.to_csv(out_dir / "D1_observed_decision_path_frequency.csv", index=False)
    atomic_write_json(summary, out_dir / "D1_governance_stability_summary.json")
    return realizations, per_client, client_summary, pairwise, cohort_freq, summary


# ======================================================================================
# D2 — 10 PREREGISTERED EVIDENCE REALIZATIONS x 3 TRAINING SEEDS
# ======================================================================================

def summarize_predictive_by_evidence(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "roc_auc_ovr_macro", "f1_macro", "accuracy", "precision_macro", "recall_macro",
        "runtime_s", "communication_mib", "optimizer_steps", "participants",
        "estimated_energy_wh", "estimated_co2_kg", "estimated_total_cost_usd",
    ]
    rows = []
    tadp = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    for eseed, d in tadp.groupby("evidence_seed", sort=True):
        row = {
            "evidence_seed": int(eseed),
            "scenario": "TADP-VR Federated",
            "n_training_seeds": int(len(d)),
            "selected_k": int(d["selected_k"].iloc[0]),
            "admitted_clients": str(d["selected_client_ids"].iloc[0]),
            "cohort_hash": str(d["cohort_hash"].iloc[0]),
        }
        for m in metrics:
            if m not in d.columns:
                continue
            x = pd.to_numeric(d[m], errors="coerce").dropna().to_numpy(dtype=float)
            if not len(x):
                continue
            lo, hi = ci95_mean(x)
            row[f"{m}_mean"] = float(np.mean(x))
            row[f"{m}_sd"] = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
            row[f"{m}_ci95_low"] = lo
            row[f"{m}_ci95_high"] = hi
        rows.append(row)

    fed = perf[perf["scenario"].eq("Vanilla FedAvg")].copy()
    if len(fed):
        row = {
            "evidence_seed": -1,
            "scenario": "Vanilla FedAvg",
            "n_training_seeds": int(len(fed)),
            "selected_k": K_SUBMISSIONS,
            "admitted_clients": ";".join(list("ABCDEFGHIJ")),
            "cohort_hash": "ALL_CLIENTS",
        }
        for m in metrics:
            if m not in fed.columns:
                continue
            x = pd.to_numeric(fed[m], errors="coerce").dropna().to_numpy(dtype=float)
            if not len(x):
                continue
            lo, hi = ci95_mean(x)
            row[f"{m}_mean"] = float(np.mean(x))
            row[f"{m}_sd"] = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
            row[f"{m}_ci95_low"] = lo
            row[f"{m}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_tadp_vs_fedavg(perf: pd.DataFrame) -> pd.DataFrame:
    rows = []
    fed = perf[perf["scenario"].eq("Vanilla FedAvg")].copy()
    for eseed in PREDICTIVE_EVIDENCE_SEEDS:
        tadp = perf[
            perf["scenario"].eq("TADP-VR Federated")
            & perf["evidence_seed"].eq(int(eseed))
        ].copy()
        if tadp.empty:
            continue
        merged = tadp.merge(
            fed, on="training_seed", suffixes=("_tadp", "_fedavg"), validate="one_to_one"
        )
        for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
            diff = (
                merged[f"{metric}_tadp"].to_numpy(dtype=float)
                - merged[f"{metric}_fedavg"].to_numpy(dtype=float)
            )
            lo, hi = ci95_mean(diff)
            rows.append({
                "evidence_seed": int(eseed),
                "metric": metric,
                "n_pairs": int(len(diff)),
                "mean_difference_tadp_minus_fedavg": float(np.mean(diff)),
                "sd_difference": float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                "ci95_low": lo,
                "ci95_high": hi,
                "tadp_wins": int(np.sum(diff > 0)),
                "ties": int(np.sum(np.isclose(diff, 0))),
                "tadp_losses": int(np.sum(diff < 0)),
            })
    return pd.DataFrame(rows)


def across_evidence_variability(summary: pd.DataFrame) -> pd.DataFrame:
    tadp = summary[summary["scenario"].eq("TADP-VR Federated")].copy()
    rows = []
    for metric in ["roc_auc_ovr_macro", "f1_macro", "accuracy"]:
        col = f"{metric}_mean"
        if col not in tadp.columns:
            continue
        x = pd.to_numeric(tadp[col], errors="coerce").dropna().to_numpy(dtype=float)
        if not len(x):
            continue
        lo, hi = ci95_mean(x)
        rows.append({
            "metric": metric,
            "n_evidence_realizations": int(len(x)),
            "mean_across_evidence_realizations": float(np.mean(x)),
            "sd_across_evidence_realizations": float(np.std(x, ddof=1)) if len(x) > 1 else 0.0,
            "min_evidence_mean": float(np.min(x)),
            "max_evidence_mean": float(np.max(x)),
            "range_across_evidence_means": float(np.max(x) - np.min(x)),
            "ci95_low_across_evidence_means": lo,
            "ci95_high_across_evidence_means": hi,
        })
    return pd.DataFrame(rows)


def build_d2_plans(client_ids, clients_raw, dq_scores, library, out_dir: Path):
    plan_rows=[]; plan_objects={}
    for eseed in PREDICTIVE_EVIDENCE_SEEDS:
        _,assignment,gov,admitted,cohort_hash=build_evidence_realization(client_ids,dq_scores,library,eseed)
        assignment.to_csv(out_dir/f"D2_assignment_seed_{eseed}.csv",index=False); gov.to_csv(out_dir/f"D2_governance_seed_{eseed}.csv",index=False)
        step_map={cid:natural_steps(len(clients_raw[cid]),BATCH_SIZE,LOCAL_EPOCHS) for cid in admitted} if admitted else {}
        plan_objects[int(eseed)]={"admitted":list(admitted),"cohort_hash":str(cohort_hash),"step_map":step_map}
        plan_rows.append({"evidence_seed":int(eseed),"selected_k":len(admitted),"admitted_clients":";".join(admitted),"cohort_hash":str(cohort_hash),
                          "trainable":bool(admitted),"steps_per_round":sum(step_map.values()) if step_map else 0,
                          "unique_bundles_sampled":assignment["bundle_id"].nunique(),"duplicate_bundle_assignments":len(assignment)-assignment["bundle_id"].nunique()})
    plans=pd.DataFrame(plan_rows); plans["cohort_reuse_count_across_10_realizations"]=plans.groupby("cohort_hash")["cohort_hash"].transform("size")
    plans.to_csv(out_dir/"D2_preregistered_evidence_plans.csv",index=False)
    return plans,plan_objects



def run_d2_predictive(data, library, out_dir: Path):
    client_ids=list(data["client_ids"]); clients_raw=data["clients_raw"]
    plans_df,plans=build_d2_plans(client_ids,clients_raw,data["dq_scores"],library,out_dir)
    empty_seeds = plans_df.loc[
        ~plans_df["trainable"], "evidence_seed"
    ].astype(int).tolist()
    if empty_seeds:
        print(
            "NOTE: preregistered evidence realization(s) with zero admitted "
            f"clients: {empty_seeds}. Governance results are retained; "
            "predictive training is skipped because no governance-eligible "
            "cohort exists."
        )

    unique_cohorts={}
    for eseed,p in plans.items():
        if not p["admitted"]: continue
        key=p["cohort_hash"]; clients=sorted(p["admitted"])
        if key not in unique_cohorts: unique_cohorts[key]={"clients":clients,"evidence_seeds":[]}
        unique_cohorts[key]["evidence_seeds"].append(int(eseed))
    views={"ALL_CLIENTS":build_post_selection_training_view(clients_raw,data["test_df"],sorted(client_ids),"ALL_CLIENTS")}
    for key,obj in unique_cohorts.items(): views[key]=build_post_selection_training_view(clients_raw,data["test_df"],obj["clients"],f"COHORT_{key}")
    pd.DataFrame([{"view_key":k,"selected_k":len(v["participating_clients"]),"selected_clients":";".join(v["participating_clients"]),
                   "input_dim":v["input_dim"],"preprocessor_sha256":v["preprocessor_sha256"],"class_weights_json":json.dumps(v["class_weights"],sort_keys=True)}
                  for k,v in views.items()]).to_csv(out_dir/"D2_optionA_preprocessing_audit.csv",index=False)
    reuse=pd.DataFrame([{"cohort_hash":k,"admitted_clients":";".join(o["clients"]),"selected_k":len(o["clients"]),
                         "evidence_realization_count":len(o["evidence_seeds"]),"evidence_seeds":";".join(map(str,sorted(o["evidence_seeds"]))) }
                        for k,o in unique_cohorts.items()]); reuse.to_csv(out_dir/"D2_unique_cohort_reuse_plan.csv",index=False)
    perf_unique_path=out_dir/"D2_unique_training_runs_checkpoint.csv"; checkpoint_state=out_dir/"D2_checkpoint_state.json"
    completed=set(load_checkpoint_state(checkpoint_state).get("completed",[])); total_jobs=len(TRAINING_RUN_SEEDS)*(1+len(unique_cohorts)); job=0
    for run_idx,tseed in enumerate(TRAINING_RUN_SEEDS,start=1):
        for view_key,scenario,role in [("ALL_CLIENTS","Vanilla FedAvg","EVIDENCE_INDEPENDENT_CONTEXTUAL_REFERENCE")]+[(k,"TADP-VR Federated","UNIQUE_EVIDENCE_COHORT") for k in sorted(unique_cohorts)]:
            job += 1; key=f"seed{tseed}|{view_key}|{scenario}"; view=views[view_key]
            if key in completed: continue
            seed_everything(tseed); build_model_fn=lambda dim=view["input_dim"]: build_diabetes_model(dim,lr=LEARNING_RATE)
            base=build_model_fn(); initial_weights=[np.array(w,copy=True) for w in base.get_weights()]; initial_hash=sha256_weights(initial_weights); del base; tf.keras.backend.clear_session()
            selected=view["participating_clients"]
            step_map={cid:natural_steps(len(clients_raw[cid]),BATCH_SIZE,LOCAL_EPOCHS) for cid in selected}
            monitor=build_train_monitor_subset(view["client_arrays"],max_samples=4096,seed=99137)
            result=federated_train(build_model_fn,initial_weights,[selected[:] for _ in range(NUM_ROUNDS_FL)],view["client_arrays"],view["X_test"],view["y_test"],3,
                                   BATCH_SIZE,LOCAL_EPOCHS,view["class_weights"],tseed,[dict(step_map) for _ in range(NUM_ROUNDS_FL)],False,0.0,monitor,
                                   {"scenario":scenario,"run_idx":run_idx,"run_total":len(TRAINING_RUN_SEEDS),"scenario_idx":job,"scenario_total":total_jobs,"overall_idx":job,"overall_total":total_jobs})
            row=result_row(run_idx,tseed,scenario,result,initial_hash); row.update({"training_seed":tseed,"cohort_hash":view_key,"selected_client_ids":";".join(selected),"selected_k":len(selected),
                 "evidence_seed":-1 if view_key=="ALL_CLIENTS" else min(unique_cohorts[view_key]["evidence_seeds"]),"comparison_role":role,
                 "input_dim":view["input_dim"],"preprocessor_sha256":view["preprocessor_sha256"],"preprocessing_fit_client_ids":";".join(selected)})
            upsert_csv(row,perf_unique_path,["training_seed","cohort_hash","scenario"]); mark_checkpoint_complete(checkpoint_state,key); completed.add(key)
            del result; tf.keras.backend.clear_session(); gc.collect()
    perf_unique=pd.read_csv(perf_unique_path)
    # expand each unique TADP cohort result back to its preregistered evidence seeds
    rows=[]
    for _,r in perf_unique.iterrows():
        if r["scenario"]=="Vanilla FedAvg": rows.append(dict(r))
        else:
            for eseed in unique_cohorts[str(r["cohort_hash"])]["evidence_seeds"]:
                rr=dict(r); rr["evidence_seed"]=int(eseed); rows.append(rr)
    perf=pd.DataFrame(rows); perf.to_csv(out_dir/"D2_performance_metrics.csv",index=False)
    summary=summarize_predictive_by_evidence(perf); summary.to_csv(out_dir/"D2_predictive_summary_by_evidence.csv",index=False)
    paired=paired_tadp_vs_fedavg(perf); paired.to_csv(out_dir/"D2_paired_TADP_vs_FedAvg.csv",index=False)
    variability=across_evidence_variability(summary); variability.to_csv(out_dir/"D2_across_evidence_variability.csv",index=False)
    # W0 parity is shape-aware under Option A.
    parity=perf_unique.groupby(["training_seed","input_dim"])["initial_weights_sha256"].nunique().reset_index(name="unique_w0_hashes")
    parity["pass"]=parity["unique_w0_hashes"].eq(1); parity.to_csv(out_dir/"D2_shape_aware_initialization_parity.csv",index=False)
    audit={
        "preregistered_evidence_realizations": int(len(PREDICTIVE_EVIDENCE_SEEDS)),
        "trainable_evidence_realizations": int(plans_df["trainable"].sum()),
        "zero_admitted_realizations": int((~plans_df["trainable"]).sum()),
        "zero_admitted_evidence_seeds": empty_seeds,
        "unique_nonempty_admitted_cohorts": int(len(unique_cohorts)),
        "training_seeds": list(TRAINING_RUN_SEEDS),
        "fedavg_actual_model_fits": int(len(TRAINING_RUN_SEEDS)),
        "tadp_actual_model_fits_after_cohort_reuse": int(
            len(unique_cohorts) * len(TRAINING_RUN_SEEDS)
        ),
        "tadp_evidence_specific_result_rows": int(
            plans_df["trainable"].sum() * len(TRAINING_RUN_SEEDS)
        ),
        "random_k_repeated_in_experiment_d": False,
        "random_k_reason": (
            "Matched Random-K is already evaluated in Experiment A1; "
            "Experiment D isolates documentary-evidence assignment sensitivity."
        ),
        "optionA_post_selection_preprocessing": True,
        "class_weights_from_participating_train_only": True,
        "unselected_clients_in_model_preprocessing": False,
        "test_used_for_fit": False,
        "dq_governance_reference_uses_raw_train_candidate_universe": True,
        "shape_aware_initialization_parity_pass": bool(parity["pass"].all()),
        "initialization_note": (
            "W0 hashes are comparable only when input dimensions match; "
            "Option-A cohort-specific vocabularies can legitimately change input_dim."
        ),
    }
    atomic_write_json(audit,out_dir/"D2_predictive_reuse_audit.json")
    return plans_df,perf,summary,paired,variability,audit



# ======================================================================================
# STARTUP / MAIN
# ======================================================================================

def startup_self_check():
    policy_self_test(); library,bundles,factors=build_documentary_bundle_library()
    if len(library)!=120 or len(bundles)!=120 or len(factors)!=120*20: raise RuntimeError("120-bundle final-policy library self-test failed")
    return True



STARTUP_SELF_CHECK_PASS = startup_self_check()


def main():
    print_banner(EXPERIMENT_VERSION)
    print("Final policy: evidence validity -> dimension floor -> HPS bands -> critical-dimension HPS_review -> ACCEPT/REJECT")
    print("Option A: participant selection precedes model preprocessing/class-weight fitting")
    root=choose_experiment_root("TADP_EXPERIMENT_D_"+EXPERIMENT_VERSION,use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS)
    design={"experiment_version":EXPERIMENT_VERSION,"k_submissions":K_SUBMISSIONS,"training_seeds":TRAINING_RUN_SEEDS,"fl_rounds":NUM_ROUNDS_FL,
            "documentary_library_size":DOCUMENTARY_LIBRARY_SIZE,"branch_families":BRANCH_FAMILIES,"d1_governance_realizations":len(GOVERNANCE_SWEEP_SEEDS),
            "d2_predictive_evidence_realizations":len(PREDICTIVE_EVIDENCE_SEEDS),"preprocessing":"Option A: fit after cohort selection on participating TRAIN only",
            "final_policy":{"evidence_validity":"all 28 factors present, finite, >0","dimension_floor":DIMENSION_MIN_FLOOR,"hps_reject_below":GOOD_CUT,
                            "direct_accept_at_or_above":HIGH_CUT,"hps_review_midpoint":REVIEW_MIDPOINT,
                            "critical_review_dimensions":CRITICAL_REVIEW_DIMS,"critical_review_weights":CRITICAL_REVIEW_WEIGHTS}}
    atomic_write_json(design,root/"experiment_design.json")
    library,bundles,factors=build_documentary_bundle_library(); bundles.to_csv(root/"documentary_library_120_bundles.csv",index=False); factors.to_csv(root/"documentary_library_120x20_factor_rows.csv",index=False)
    csv_path=locate_diabetes_csv(); data=prepare_main_k10(csv_path); data["partition_table"].to_csv(root/"frozen_client_partition_K10_A_to_J.csv",index=False); data["dq_audit"].to_csv(root/"frozen_DQ_audit_K10.csv",index=False); atomic_write_json(data["leakage_audit"],root/"leakage_audit.json")
    d1=run_d1_governance_sensitivity(data["client_ids"],data["dq_scores"],library,root)
    plans,perf,pred_summary,paired,variability,d2_audit=run_d2_predictive(data,library,root)
    print_banner("EXPERIMENT D — FINAL DESIGN AUDITS")
    print("28-factor evidence-validity policy: PASS"); print("HPS_review D1/D2/D5/D6: PASS"); print("Option-A cohort-specific preprocessing: PASS"); print("Held-out TEST fit leakage: PASS")
    return root



if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)


📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US (1).csv
✅ Uploaded: diabetes_130US (1).csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)

TADP-D-FINAL-A1A2BC-28F-HPSREVIEW-OPTIONA-K10-500GOV-10EVID-3SEED-4ROUND
Final policy: evidence validity -> dimension floor -> HPS bands -> critical-dimension HPS_review -> ACCEPT/REJECT
Option A: participant selection precedes model preprocessing/class-weight fitting
Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_D_TADP-D-FINAL-A1A2BC-28F-HPSREVIEW-OPTIONA-K10-500GOV-10EVID-3SEED-4ROUND
D1 governance-only assignments: 50/500 complete
D1 governance-only assignments: 100/500 complete
D1 governance-only assignments: 150/500 complete
D1 governance-only assignments: 200/500 complete
D1 governance-only assignments: 250/500 complete
D1 governance-only assignments: 300/500 complete
D1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>